In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_path = "/content/drive/MyDrive/dataset"

In [ ]:
from PIL import Image
import os

def clean(folder):
    for f in os.listdir(folder):
        p = os.path.join(folder, f)
        try:
            with Image.open(p) as img:
                img.verify()
        except:
            os.remove(p)

clean(os.path.join(dataset_path,"normal"))
clean(os.path.join(dataset_path,"abnormal"))

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf

img_size = 128
batch_size = 32

normal_path = dataset_path + "/normal"

normal_ds = tf.keras.preprocessing.image_dataset_from_directory(
    normal_path,
    labels=None,
    image_size=(img_size, img_size),
    batch_size=batch_size
)

Found 5386 files.


In [ ]:
normal_ds = normal_ds.map(lambda x: x/255.0)

In [ ]:
abnormal_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path + "/abnormal",
    labels=None,
    image_size=(img_size, img_size),
    batch_size=batch_size
)

abnormal_ds = abnormal_ds.map(lambda x: x/255.0)

Found 180 files.


In [ ]:
model = models.Sequential([
    layers.Input(shape=(128,128,3)),

    layers.Conv2D(32,3,activation='relu',padding='same'),
    layers.MaxPooling2D(2),

    layers.Conv2D(64,3,activation='relu',padding='same'),
    layers.MaxPooling2D(2),

    layers.Conv2D(64,3,activation='relu',padding='same'),

    layers.UpSampling2D(2),
    layers.Conv2D(64,3,activation='relu',padding='same'),

    layers.UpSampling2D(2),
    layers.Conv2D(32,3,activation='relu',padding='same'),

    layers.Conv2D(3,3,activation='sigmoid',padding='same')
])

model.compile(optimizer='adam', loss='mse')

In [ ]:
import tensorflow as tf

# Parameters
img_size = 128
batch_size = 32

# Path to ONLY normal images
normal_path = dataset_path + "/normal"

# 🔹 Load images
normal_ds = tf.keras.preprocessing.image_dataset_from_directory(
    normal_path,
    labels=None,
    image_size=(img_size, img_size),
    batch_size=batch_size
)

# 🔹 Normalize images (0–1 range)
normal_ds = normal_ds.map(lambda x: x / 255.0)

# 🔹 Convert to (input, target) format → REQUIRED for autoencoder
normal_ds_for_autoencoder = normal_ds.map(lambda x: (x, x))

# 🔹 Prefetch for faster training (important but simple)
AUTOTUNE = tf.data.AUTOTUNE
normal_ds_for_autoencoder = normal_ds_for_autoencoder.prefetch(buffer_size=AUTOTUNE)

# 🔹 VERIFY dataset (MANDATORY CHECK)
for batch in normal_ds_for_autoencoder.take(1):
    print("Input shape:", batch[0].shape)
    print("Target shape:", batch[1].shape)

Found 5386 files.
Input shape: (32, 128, 128, 3)
Target shape: (32, 128, 128, 3)


In [ ]:
history = model.fit(normal_ds_for_autoencoder, epochs=10)

Epoch 1/10
169/169 ━━━━━━━━━━━━━━━━━━━━ 1103s 6s/step - loss: 0.0089
Epoch 2/10
169/169 ━━━━━━━━━━━━━━━━━━━━ 1053s 6s/step - loss: 0.0028
Epoch 3/10
169/169 ━━━━━━━━━━━━━━━━━━━━ 1050s 6s/step - loss: 0.0024
Epoch 4/10
 57/169 ━━━━━━━━━━━━━━━━━━━━ 12:33 7s/step - loss: 0.0022

In [ ]:
def get_loss(dataset):
    losses = []
    for batch in dataset:
        recon = model.predict(batch)
        loss = np.mean((batch - recon)**2, axis=(1,2,3))
        losses.extend(loss)
    return np.array(losses)

normal_loss = get_loss(normal_ds)
abnormal_loss = get_loss(abnormal_ds)

In [ ]:
plt.hist(normal_loss, bins=50, alpha=0.5, label="Normal")
plt.hist(abnormal_loss, bins=50, alpha=0.5, label="Abnormal")
plt.legend()
plt.title("Reconstruction Error Distribution")
plt.show()

In [ ]:
from tensorflow.keras.models import Model
import numpy as np
import tensorflow as tf

# 🔴 STEP 1: Create an explicit Input tensor for the encoder
# This avoids relying on model.input which might not be consistently exposed for Sequential models.
input_for_encoder = tf.keras.Input(shape=(img_size, img_size, 3))

# 🔴 STEP 2: Pass the new Input tensor through the original model's layers
# to get the output tensor at the desired intermediate point (layer 4).
x = input_for_encoder
encoder_output_tensor = None

# Iterate through the original model's layers to build the encoder part
for i, layer in enumerate(model.layers):
    x = layer(x)
    if i == 4:  # model.layers[4] is the output layer for our encoder
        encoder_output_tensor = x
        break # Stop after we've found the desired output

# 🔴 STEP 3: Create the encoder using the new Input and the derived output tensor
encoder = Model(inputs=input_for_encoder, outputs=encoder_output_tensor)

print("Encoder created successfully")

# 🔴 STEP 4: Feature extraction function
def extract_features(dataset):
    features = []

    for batch in dataset:
        f = encoder.predict(batch)
        f = f.reshape(f.shape[0], -1) # Flatten the features for each image in the batch
        features.extend(f)

    return np.array(features)

# 🔴 STEP 5: Extract features
normal_feat = extract_features(normal_ds)
abnormal_feat = extract_features(abnormal_ds)

print("Feature extraction done")
print("Normal:", normal_feat.shape)
print("Abnormal:", abnormal_feat.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Calculate the mean feature value for each sample
normal_feat_means = np.mean(normal_feat, axis=1)
abnormal_feat_means = np.mean(abnormal_feat, axis=1)

# Create a DataFrame for plotting
plot_data = pd.DataFrame({
    'Feature Mean': np.concatenate([normal_feat_means, abnormal_feat_means]),
    'Type': ['Normal'] * len(normal_feat_means) + ['Abnormal'] * len(abnormal_feat_means)
})

# Create the box plot
plt.figure(figsize=(8, 6))
sns.boxplot(x='Type', y='Feature Mean', data=plot_data)
plt.title('Distribution of Mean Feature Values for Normal and Abnormal Samples')
plt.ylabel('Mean Feature Value')
plt.xlabel('Sample Type')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from sklearn.manifold import TSNE

X = np.vstack([normal_feat, abnormal_feat])
y = np.array([0]*len(normal_feat) + [1]*len(abnormal_feat))

tsne = TSNE(n_components=2, random_state=42)
X_emb = tsne.fit_transform(X)

plt.figure(figsize=(8,6))
plt.scatter(X_emb[y==0,0], X_emb[y==0,1], alpha=0.5, label="Normal")
plt.scatter(X_emb[y==1,0], X_emb[y==1,1], alpha=0.5, label="Abnormal")
plt.legend()
plt.title("Latent Feature Space")
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

# Calculate the silhouette score
sil_score = silhouette_score(X_emb, y)

print(f"Silhouette Score for t-SNE clusters: {sil_score:.4f}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(normal_feat)
intra_class = np.mean(sim)
print("Intra-class similarity:", intra_class)

In [ ]:
normal_center = np.mean(normal_feat, axis=0)
abnormal_center = np.mean(abnormal_feat, axis=0)

inter_dist = np.linalg.norm(normal_center - abnormal_center)
print("Inter-class distance:", inter_dist)

In [ ]:
from sklearn.metrics import silhouette_score

labels = np.array([0]*len(normal_feat) + [1]*len(abnormal_feat))
score = silhouette_score(X, labels)

print("Silhouette Score:", score)

In [ ]:
from sklearn.metrics import roc_auc_score

y_true = np.array([0]*len(normal_loss) + [1]*len(abnormal_loss))
scores = np.concatenate([normal_loss, abnormal_loss])

auc = roc_auc_score(y_true, scores)
print("AUROC:", auc)

In [ ]:
plt.boxplot([normal_loss, abnormal_loss], labels=["Normal","Abnormal"])
plt.title("Reconstruction Error Comparison")
plt.ylabel("Error")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

y_true = np.array([0]*len(normal_loss) + [1]*len(abnormal_loss))
scores = np.concatenate([normal_loss, abnormal_loss])

fpr, tpr, _ = roc_curve(y_true, scores)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Deviation Detection)")
plt.show()

In [ ]:
normal_center = np.mean(normal_feat, axis=0)

normal_dist = np.linalg.norm(normal_feat - normal_center, axis=1)
abnormal_dist = np.linalg.norm(abnormal_feat - normal_center, axis=1)

plt.hist(normal_dist, bins=50, alpha=0.5, label="Normal")
plt.hist(abnormal_dist, bins=50, alpha=0.5, label="Abnormal")
plt.legend()
plt.title("Distance from Normal Feature Center")
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(normal_feat[:200])  # subset to avoid overload

plt.imshow(sim_matrix)
plt.colorbar()
plt.title("Feature Similarity (Normal Images)")
plt.show()

In [ ]:
from sklearn.metrics import silhouette_samples

labels = np.array([0]*len(normal_feat) + [1]*len(abnormal_feat))
X = np.vstack([normal_feat, abnormal_feat])

sil_vals = silhouette_samples(X, labels)

plt.hist(sil_vals, bins=50)
plt.title("Silhouette Distribution")
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
import umap

reducer = umap.UMAP(random_state=42)
X_umap = reducer.fit_transform(X)

plt.figure(figsize=(8,6))
plt.scatter(X_umap[y==0,0], X_umap[y==0,1], alpha=0.5, label="Normal")
plt.scatter(X_umap[y==1,0], X_umap[y==1,1], alpha=0.5, label="Abnormal")
plt.legend()
plt.title("UMAP Projection of Latent Features")
plt.show()

In [ ]:
import numpy as np

def stats(arr):
    return [
        np.mean(arr),
        np.std(arr),
        np.median(arr),
        np.min(arr),
        np.max(arr)
    ]

normal_stats = stats(normal_loss)
abnormal_stats = stats(abnormal_loss)

print("Normal:", normal_stats)
print("Abnormal:", abnormal_stats)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# take one batch
for batch in abnormal_ds.take(1):
    recon = model.predict(batch)

    diff = np.abs(batch - recon)

    plt.subplot(1,3,1)
    plt.imshow(batch[0])
    plt.title("Original")

    plt.subplot(1,3,2)
    plt.imshow(recon[0])
    plt.title("Reconstructed")

    plt.subplot(1,3,3)
    plt.imshow(diff[0])
    plt.title("Difference Map")

    plt.show()

In [ ]:
import seaborn as sns

sns.kdeplot(X_emb[y==0,0], label="Normal")
sns.kdeplot(X_emb[y==1,0], label="Abnormal")

plt.legend()
plt.title("Feature Density Distribution")
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics.pairwise import euclidean_distances

dist_matrix = euclidean_distances(normal_feat[:200])

sns.heatmap(dist_matrix)
plt.title("Distance Matrix (Normal Images)")
plt.show()

In [ ]:
import numpy as np

def compute_stats(arr):
    return {
        "Mean": np.mean(arr),
        "Std": np.std(arr),
        "Median": np.median(arr),
        "Min": np.min(arr),
        "Max": np.max(arr)
    }

normal_stats = compute_stats(normal_loss)
abnormal_stats = compute_stats(abnormal_loss)

print("Normal:", normal_stats)
print("Abnormal:", abnormal_stats)

In [ ]:
from sklearn.metrics import roc_auc_score

y_true = np.array([0]*len(normal_loss) + [1]*len(abnormal_loss))
scores = np.concatenate([normal_loss, abnormal_loss])

auc = roc_auc_score(y_true, scores)
print("AUROC:", auc)

In [ ]:
from scipy.stats import mannwhitneyu

stat, p_value = mannwhitneyu(normal_loss, abnormal_loss)
print("p-value:", p_value)

In [ ]:
normal_dist = np.linalg.norm(normal_feat - normal_center, axis=1)
abnormal_dist = np.linalg.norm(abnormal_feat - normal_center, axis=1)

print("Mean normal distance:", np.mean(normal_dist))
print("Mean abnormal distance:", np.mean(abnormal_dist))

In [ ]:
threshold = np.mean(normal_loss) + 2*np.std(normal_loss)
print("Threshold:", threshold)

In [ ]:
from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(normal_loss, abnormal_loss)
print(p)